<a href="https://colab.research.google.com/github/Roshan5105labs/crisis-logistics-env/blob/main/crisis_logistics_env/notebooks/logiflow_grpo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LogiFlow-RL — GRPO Training (Phase 2)

**One-click reproducible training notebook for judges.**

This notebook clones the repo, installs dependencies, and runs the production training script (`train_grpo.py`) which:
1. Validates the reward function before training
2. Evaluates baseline policies (round_robin, heuristic) BEFORE training
3. Runs GRPO with TRL + LoRA on Qwen2.5-0.5B-Instruct
4. Evaluates the trained model AFTER training
5. Saves reward curve PNG, before/after CSVs, and improvement JSON


In [1]:
# === Cell 1: Clone repo and install dependencies ===
import os, subprocess, sys
from pathlib import Path

# Update this if your repo URL is different
REPO_URL = "https://github.com/Roshan5105labs/crisis-logistics-env.git"
REPO_ROOT = Path("/content/crisis-logistics-env")

# Clone (or pull if already cloned)
if not REPO_ROOT.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_ROOT)])
else:
    subprocess.call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"])

# Find the package directory (handles both flat and nested layouts)
candidates = [
    REPO_ROOT / "crisis_logistics_env",
    REPO_ROOT,  # if the repo IS the package
]
PKG_DIR = next((p for p in candidates if (p / "server").exists() and (p / "tasks.py").exists()), None)
if PKG_DIR is None:
    raise FileNotFoundError(f"Could not find package in {REPO_ROOT}")

# Add the parent of the package to sys.path so 'crisis_logistics_env' is importable
sys.path.insert(0, str(PKG_DIR.parent))
os.chdir(PKG_DIR)
print(f"Working from: {PKG_DIR}")
print(f"sys.path[0]: {sys.path[0]}")

Working from: /content/crisis-logistics-env/crisis_logistics_env
sys.path[0]: /content/crisis-logistics-env


In [2]:
# === Cell 2: Install pinned dependencies ===
# Pin TRL to a known-good version. Newer TRL has GRPOConfig at trl.GRPOConfig.
# If trl 0.12 doesn't have it, fallback to 0.14+
!pip install -q "transformers==4.47.1" "trl==0.14.0" "accelerate==0.34.0" "peft==0.13.2" "openenv.core"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# === Cell 3: Verify the environment imports correctly ===
from crisis_logistics_env.server.crisis_logistics_env_environment import CrisisLogisticsEnvironment
from crisis_logistics_env.tasks import list_tasks

env = CrisisLogisticsEnvironment()
obs = env.reset(task_id="easy")
print(f"✓ Environment loaded — task: {obs.task_id}")
print(f"✓ Available tasks: {[t.task_id for t in list_tasks()]}")
print(f"✓ Observation has {len(obs.node_loads)} nodes (expected 12)")
print(f"✓ Visible nodes: {obs.visible_node_ids}")
print(f"✓ First step source: {obs.pending_source_node}, volume: {obs.incoming_load}")

assert len(obs.node_loads) == 12, "Expected 12-node network"
print("\n✓ Environment sanity check PASSED")

✓ Environment loaded — task: easy
✓ Available tasks: ['easy', 'medium', 'hard']
✓ Observation has 12 nodes (expected 12)
✓ Visible nodes: [0, 1, 2, 3, 4, 5, 7, 8, 9, 10]
✓ First step source: 0, volume: 9.0

✓ Environment sanity check PASSED


In [4]:
import os, torch, gc

# Tell PyTorch to use expandable memory segments (reduces fragmentation)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Kill any leftover models from previous cells
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free, total = torch.cuda.mem_get_info()
print(f"Free: {free/1e9:.2f} GB / Total: {total/1e9:.2f} GB")
# You need at least 3 GB free before starting

Free: 15.53 GB / Total: 15.64 GB


In [ ]:
# === Cell 4: Run the production GRPO training script ===
# This runs train_grpo.py which:
#   1. Builds training dataset from environment rollouts
#   2. Sanity-checks the reward function (refuses to start if broken)
#   3. Evaluates round_robin + heuristic baselines BEFORE training
#   4. Runs GRPO training (LoRA on Qwen2.5-0.5B-Instruct)
#   5. Evaluates the trained model AFTER training
#   6. Saves all artifacts

# Reduce max_steps if you're short on time. 200 is the sweet spot for visible learning.
# Use --max-steps 80 for a quick smoke test (~15 min).
!PYTORCH_ALLOC_CONF=expandable_segments:True \
 python train_grpo.py \
    --model-name "Qwen/Qwen2.5-0.5B-Instruct" \
    --max-steps 80 \
    --num-generations 2 \
    --samples-per-task 30 \
    --output-dir "outputs/logiflow-grpo-script"

2026-04-26 03:32:14.073892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777174334.096810    1532 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777174334.104822    1532 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777174334.124156    1532 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777174334.124234    1532 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777174334.124239    1532 computation_placer.cc:177] computation placer alr

In [ ]:
import sys
sys.path.insert(0, "outputs/logiflow-grpo-script")

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16, device_map="auto"
)
adapter_path = "outputs/logiflow-grpo-script"
model = PeftModel.from_pretrained(base, adapter_path)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
if tok.pad_token is None: tok.pad_token = tok.eos_token

# Generate one decision on a hard task
from crisis_logistics_env.server.crisis_logistics_env_environment \
    import CrisisLogisticsEnvironment
env = CrisisLogisticsEnvironment()
obs = env.reset(task_id="hard")
for _ in range(8): obs = env.step(__import__("crisis_logistics_env"
    ).server.crisis_logistics_env_environment.choose_network_action(obs))

prompt = (
    f"Task: {env.task.title}\\n"
    f"Visible nodes: {obs.visible_node_ids}\\n"
    f"Loads: {obs.observed_node_loads}\\n"
    f"Active disruptions: {obs.active_disruptions}\\n"
    f"Incoming: source={obs.pending_source_node}, vol={obs.incoming_load}\\n"
    "Return JSON: {reasoning, source_node, dest_node, shipment_volume}"
)

inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=150, do_sample=False,
                     pad_token_id=tok.eos_token_id)
text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== Trained agent's reasoning ===")
print(text)

In [ ]:
# === Cell 5: Display training artifacts ===
from pathlib import Path
import json
from IPython.display import Image, display

ARTIFACTS = Path("outputs/logiflow-grpo-script/artifacts")

if not ARTIFACTS.exists():
    print("⚠ Artifacts folder not found. Did training complete?")
    print(f"Looked at: {ARTIFACTS.resolve()}")
else:
    print("=== Generated artifacts ===")
    for f in sorted(ARTIFACTS.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:40s}  {size_kb:>8.1f} KB")
    print()

    # Show reward curve
    curve_path = ARTIFACTS / "reward_curve.png"
    if curve_path.exists():
        print("=== Reward Curve ===")
        display(Image(str(curve_path)))

    # Show improvement summary
    improvement_path = ARTIFACTS / "improvement.json"
    if improvement_path.exists():
        print("\n=== Improvement Summary ===")
        with open(improvement_path) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))

    # Show evaluation summary
    eval_path = ARTIFACTS / "evaluation_summary.json"
    if eval_path.exists():
        print("\n=== Evaluation Summary (before vs after) ===")
        with open(eval_path) as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))

In [ ]:
# === Cell 6: Before vs After comparison table ===
import pandas as pd
from pathlib import Path

ARTIFACTS = Path("outputs/logiflow-grpo-script/artifacts")

before_csv = ARTIFACTS / "evaluation_before.csv"
after_csv = ARTIFACTS / "evaluation_after.csv"

if before_csv.exists() and after_csv.exists():
    before = pd.read_csv(before_csv)
    after = pd.read_csv(after_csv)

    print("=== BEFORE TRAINING ===")
    print(before.to_string(index=False))
    print("\n=== AFTER TRAINING ===")
    print(after.to_string(index=False))

    # Build comparison if model policy exists in both
    if 'policy' in before.columns and 'policy' in after.columns:
        merged = before.merge(after, on=['policy', 'task_id'], suffixes=('_before', '_after'))
        merged['score_delta'] = merged['score_after'] - merged['score_before']
        print("\n=== DELTA ===")
        print(merged[['policy', 'task_id', 'score_before', 'score_after', 'score_delta']].to_string(index=False))
else:
    print("Evaluation CSVs not found — check training logs.")

In [ ]:
# === GRAPH 1: Training Reward Curve ===
# Reads from reward_curve.csv saved by train_grpo.py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Image

ARTIFACTS = Path('outputs/logiflow-grpo-script/artifacts')

if not ARTIFACTS.exists():
    print('Artifacts not found. Run Cell 4 (training) first.')
else:
    curve_csv = ARTIFACTS / 'reward_curve.csv'
    curve_png = ARTIFACTS / 'reward_curve.png'

    if curve_png.exists():
        # If train_grpo.py already saved a PNG, show it directly
        print('=== Training Reward Curve ===')
        display(Image(str(curve_png)))

    if curve_csv.exists():
        # Re-plot from CSV with better styling
        df = pd.read_csv(curve_csv)
        history = df['mean_reward'].tolist()
        steps   = df['step'].tolist()

        def smooth(vals, w=5):
            return [sum(vals[max(0,i-w):i+1])/len(vals[max(0,i-w):i+1])
                    for i in range(len(vals))]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.patch.set_facecolor('#0f1117')
        fig.suptitle('GRPO Training — LogiFlow-RL (Qwen2.5-0.5B + LoRA)',
                     color='white', fontsize=14, fontweight='bold', y=1.02)

        PANEL='#1a1f2e'; BLUE='#2980b9'; AMBER='#f39c12'; TEXT='#e8edf5'; MUTED='#5a6a7a'

        # Left: full reward curve
        ax = axes[0]
        ax.set_facecolor(PANEL)
        ax.fill_between(steps, history, alpha=0.12, color=BLUE)
        ax.plot(steps, history, color=BLUE, alpha=0.35, linewidth=1, label='Raw')
        if len(history) > 5:
            ax.plot(steps, smooth(history), color=BLUE, linewidth=2.5, label='Smoothed')
        ax.axhline(0.5, color=AMBER, linestyle='--', linewidth=1, alpha=0.6,
                   label='0.5 baseline')
        ax.set_xlabel('Logging step', color=TEXT, fontsize=11)
        ax.set_ylabel('Mean reward (0–1)', color=TEXT, fontsize=11)
        ax.set_title('Full training run', color=TEXT, fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.tick_params(colors=TEXT)
        ax.spines[:].set_color(MUTED)
        ax.legend(fontsize=9, facecolor=PANEL, labelcolor=TEXT)
        ax.grid(True, alpha=0.12, color=MUTED)

        # Right: rolling average clearly showing trend
        ax2 = axes[1]
        ax2.set_facecolor(PANEL)
        if len(history) > 10:
            s3  = smooth(history, 3)
            s10 = smooth(history, 10)
            ax2.plot(steps, s3,  color='#5acdff', linewidth=1.5,
                     alpha=0.7, label='Short smooth (w=3)')
            ax2.plot(steps, s10, color=BLUE,      linewidth=2.5,
                     label='Long smooth (w=10)')
            # Shade learning phase
            mid = len(steps)//2
            ax2.axvspan(0, steps[mid], alpha=0.05, color=AMBER, label='Early phase')
            ax2.axvspan(steps[mid], steps[-1], alpha=0.05, color='#27ae60', label='Late phase')
        ax2.set_xlabel('Logging step', color=TEXT, fontsize=11)
        ax2.set_ylabel('Smoothed mean reward', color=TEXT, fontsize=11)
        ax2.set_title('Learning trend', color=TEXT, fontsize=11)
        ax2.set_ylim(0, 1.05)
        ax2.tick_params(colors=TEXT)
        ax2.spines[:].set_color(MUTED)
        ax2.legend(fontsize=8, facecolor=PANEL, labelcolor=TEXT)
        ax2.grid(True, alpha=0.12, color=MUTED)

        plt.tight_layout()
        out = ARTIFACTS / 'reward_curve_hd.png'
        plt.savefig(str(out), dpi=160, bbox_inches='tight',
                    facecolor='#0f1117')
        plt.show()
        print(f'Saved: {out}')
    else:
        print('reward_curve.csv not found — show png only above')


In [ ]:
# === GRAPH 2: Before vs After Score Comparison ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from pathlib import Path

ARTIFACTS = Path('outputs/logiflow-grpo-script/artifacts')
before_csv = ARTIFACTS / 'evaluation_before.csv'
after_csv  = ARTIFACTS / 'evaluation_after.csv'

if not before_csv.exists() or not after_csv.exists():
    print('Evaluation CSVs not found. Run Cell 4 (training) first.')
else:
    before = pd.read_csv(before_csv)
    after  = pd.read_csv(after_csv)

    tasks   = sorted(before['task_id'].unique())
    n_tasks = len(tasks)

    # Extract scores per policy
    def scores(df, policy):
        sub = df[df.policy == policy]
        return [sub[sub.task_id == t]['score'].values[0]
                if len(sub[sub.task_id == t]) else 0.0 for t in tasks]

    # Collect all policies present
    all_policies = list(before['policy'].unique()) + list(after['policy'].unique())
    has_heuristic = 'heuristic' in all_policies
    has_round     = 'round_robin' in all_policies
    has_model_b   = 'model' in before['policy'].values or 'model' in before.get('policy', pd.Series()).values
    has_model_a   = 'model' in after['policy'].values

    # Build comparison rows
    score_groups = {}
    for p in ['round_robin', 'heuristic']:
        df = before if p in before['policy'].values else after
        if p in df['policy'].values:
            score_groups[p] = scores(df, p)
    if has_model_b:
        score_groups['model (before)'] = scores(before, 'model')
    if has_model_a:
        score_groups['model (after)'] = scores(after, 'model')

    PANEL='#1a1f2e'; TEXT='#e8edf5'; MUTED='#5a6a7a'
    COLORS = {
        'round_robin':    '#e74c3c',
        'heuristic':      '#f39c12',
        'model (before)': '#5acdff',
        'model (after)':  '#27ae60',
    }

    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    fig.patch.set_facecolor('#0f1117')
    fig.suptitle('Before vs After Training — Policy Comparison',
                 color='white', fontsize=14, fontweight='bold')

    for ax_idx, task in enumerate(tasks):
        ax = axes[ax_idx]
        ax.set_facecolor(PANEL)
        policies = list(score_groups.keys())
        vals = [score_groups[p][ax_idx] for p in policies]
        colors = [COLORS.get(p, '#888') for p in policies]
        x = np.arange(len(policies))
        bars = ax.bar(x, vals, color=colors, width=0.6, alpha=0.88,
                      edgecolor='white', linewidth=0.5)

        # Value labels on bars
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                    f'{val:.3f}', ha='center', va='bottom',
                    color=TEXT, fontsize=9, fontweight='bold')

        # Improvement arrow if before+after model scores exist
        if 'model (before)' in score_groups and 'model (after)' in score_groups:
            b_val = score_groups['model (before)'][ax_idx]
            a_val = score_groups['model (after)'][ax_idx]
            b_idx = policies.index('model (before)')
            a_idx = policies.index('model (after)')
            delta = a_val - b_val
            mid_x = (b_idx + a_idx) / 2
            ax.annotate('', xy=(a_idx, a_val + 0.04),
                        xytext=(b_idx, b_val + 0.04),
                        arrowprops=dict(arrowstyle='->', color='#27ae60',
                                        lw=2))
            ax.text(mid_x, max(b_val, a_val) + 0.08,
                    f'{delta:+.3f}', ha='center', color='#27ae60',
                    fontsize=10, fontweight='bold')

        ax.set_xticks(x)
        ax.set_xticklabels([p.replace('_', '\n') for p in policies],
                           color=TEXT, fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel('Score (0–1)', color=TEXT, fontsize=10)
        ax.set_title(f'Task: {task.upper()}', color=TEXT, fontsize=11,
                     fontweight='bold')
        ax.tick_params(colors=TEXT)
        ax.spines[:].set_color(MUTED)
        ax.grid(True, axis='y', alpha=0.12, color=MUTED)

    plt.tight_layout()
    out = ARTIFACTS / 'before_after_comparison.png'
    plt.savefig(str(out), dpi=160, bbox_inches='tight', facecolor='#0f1117')
    plt.show()
    print(f'Saved: {out}')


In [ ]:
# === GRAPH 3: All Metrics Panel (SLA, Bottlenecks, Invalid Actions, Retail Delivered) ===
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

ARTIFACTS = Path('outputs/logiflow-grpo-script/artifacts')
before_csv = ARTIFACTS / 'evaluation_before.csv'
after_csv  = ARTIFACTS / 'evaluation_after.csv'

if not before_csv.exists() or not after_csv.exists():
    print('Run Cell 4 first.')
else:
    before = pd.read_csv(before_csv)
    after  = pd.read_csv(after_csv)

    tasks = sorted(before['task_id'].unique())
    PANEL='#1a1f2e'; TEXT='#e8edf5'; MUTED='#5a6a7a'
    C_BEFORE='#5acdff'; C_AFTER='#27ae60'; C_HEUR='#f39c12'

    METRICS = [
        ('score',            'Overall Score',         True,  1.0),
        ('sla_rate',         'SLA Success Rate',      True,  1.0),
        ('retail_delivered', 'Retail Delivered',      True,  None),
        ('invalid_actions',  'Invalid Actions',       False, None),
        ('bottlenecks',      'Bottlenecks',           False, None),
    ]
    # Only include metrics that exist in both CSVs
    METRICS = [(m, l, hi, s) for m, l, hi, s in METRICS
               if m in before.columns and m in after.columns]

    n_metrics = len(METRICS)
    if n_metrics == 0:
        print('No matching metric columns found in CSVs.')
    else:
        fig, axes = plt.subplots(1, n_metrics, figsize=(4*n_metrics, 5))
        if n_metrics == 1: axes = [axes]
        fig.patch.set_facecolor('#0f1117')
        fig.suptitle('Detailed Metrics — Before vs After Training',
                     color='white', fontsize=14, fontweight='bold')

        for ax, (metric, label, higher_is_better, scale) in zip(axes, METRICS):
            ax.set_facecolor(PANEL)
            x = np.arange(len(tasks))
            w = 0.26

            # Collect values
            def get_metric(df, policy, metric, tasks):
                sub = df[df.policy == policy]
                return [float(sub[sub.task_id==t][metric].values[0])
                        if len(sub[sub.task_id==t]) else 0.0 for t in tasks]

            h_vals  = get_metric(after,  'heuristic', metric, tasks)
            b_vals  = get_metric(before, 'model',     metric, tasks)
            a_vals  = get_metric(after,  'model',     metric, tasks)

            bars_h = ax.bar(x - w,   h_vals, w, color=C_HEUR,   alpha=0.85,
                            label='Heuristic',    edgecolor='white', linewidth=0.4)
            bars_b = ax.bar(x,       b_vals, w, color=C_BEFORE, alpha=0.85,
                            label='Model before', edgecolor='white', linewidth=0.4)
            bars_a = ax.bar(x + w,   a_vals, w, color=C_AFTER,  alpha=0.85,
                            label='Model after',  edgecolor='white', linewidth=0.4)

            for bar in list(bars_h) + list(bars_b) + list(bars_a):
                h = bar.get_height()
                label_text = f'{h:.2f}' if h < 100 else f'{int(h)}'
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.01*max(h_vals+b_vals+a_vals+[1]),
                        label_text, ha='center', va='bottom',
                        color=TEXT, fontsize=7)

            ax.set_xticks(x)
            ax.set_xticklabels([t.upper() for t in tasks], color=TEXT, fontsize=9)
            ax.set_title(label + ('\n↑ higher = better' if higher_is_better
                                  else '\n↓ lower = better'),
                         color=TEXT, fontsize=9)
            ax.tick_params(colors=TEXT)
            ax.spines[:].set_color(MUTED)
            ax.grid(True, axis='y', alpha=0.12, color=MUTED)
            ax.legend(fontsize=7, facecolor=PANEL, labelcolor=TEXT,
                      loc='upper right')

        plt.tight_layout()
        out = ARTIFACTS / 'metrics_panel.png'
        plt.savefig(str(out), dpi=160, bbox_inches='tight', facecolor='#0f1117')
        plt.show()
        print(f'Saved: {out}')


In [ ]:
# === GRAPH 4: Improvement Summary Table (printed + saved) ===
import pandas as pd
import json
from pathlib import Path

ARTIFACTS = Path('outputs/logiflow-grpo-script/artifacts')
before_csv = ARTIFACTS / 'evaluation_before.csv'
after_csv  = ARTIFACTS / 'evaluation_after.csv'

if before_csv.exists() and after_csv.exists():
    before = pd.read_csv(before_csv)
    after  = pd.read_csv(after_csv)

    tasks = sorted(before['task_id'].unique())

    rows = []
    for task in tasks:
        def g(df, policy, col):
            sub = df[(df.policy==policy) & (df.task_id==task)]
            return float(sub[col].values[0]) if len(sub) and col in df.columns else None

        row = {'task': task}
        for p in ['round_robin', 'heuristic']:
            v = g(after, p, 'score') or g(before, p, 'score')
            if v is not None: row[f'{p}_score'] = round(v, 4)

        b = g(before, 'model', 'score')
        a = g(after,  'model', 'score')
        if b is not None: row['model_before'] = round(b, 4)
        if a is not None: row['model_after']  = round(a, 4)
        if b is not None and a is not None:
            delta = a - b
            row['delta']   = round(delta, 4)
            row['improved'] = '✓' if delta >= 0 else '✗'
        rows.append(row)

    df_summary = pd.DataFrame(rows).set_index('task')

    print('=' * 70)
    print(' LOGIFLOW-RL — SUBMISSION RESULTS SUMMARY')
    print('=' * 70)
    print(df_summary.to_string())
    print('=' * 70)

    if 'delta' in df_summary.columns:
        avg = df_summary['delta'].mean()
        print(f'\n  Average improvement: {avg:+.4f}')
        print(f'  Tasks improved:      {(df_summary["delta"] >= 0).sum()} / {len(tasks)}')

    # Save as CSV and JSON
    df_summary.to_csv(ARTIFACTS / 'results_summary.csv')
    df_summary.reset_index().to_json(
        str(ARTIFACTS / 'results_summary.json'), orient='records', indent=2)
    print(f'\n  Saved: results_summary.csv')
    print(f'  Saved: results_summary.json')

else:
    print('Run training first.')


In [ ]:
# === FINAL: List all saved artifacts and display all graphs ===
import os
from pathlib import Path
from IPython.display import Image, display, HTML

ARTIFACTS = Path('outputs/logiflow-grpo-script/artifacts')

if ARTIFACTS.exists():
    files = sorted(ARTIFACTS.iterdir())
    print('=== ALL SUBMISSION ARTIFACTS ===')
    total_kb = 0
    for f in files:
        kb = f.stat().st_size / 1024
        total_kb += kb
        print(f'  {f.name:48s} {kb:>8.1f} KB')
    print(f'  {"TOTAL":48s} {total_kb:>8.1f} KB')

    print('\n=== ALL GRAPHS ===')
    for png in sorted(ARTIFACTS.glob('*.png')):
        print(f'\n--- {png.name} ---')
        display(Image(str(png)))

    print('\n=== NEXT STEPS ===')
    print('1. Download the artifacts/ folder (Colab Files panel → right-click → Download)')
    print('2. git add artifacts/ reward_curve.png before_after_comparison.png metrics_panel.png')
    print('3. git commit -m "Add training artifacts and graphs"')
    print('4. Embed reward_curve.png in README.md like this:')
    print('   ![Reward Curve](artifacts/reward_curve.png)')
    print('5. Embed before_after_comparison.png in README.md')
    print('6. Push to HuggingFace Space: git push')
else:
    print('No artifacts found. Run Cell 4 (training) first.')


## What just happened

1. **Cloned and installed** the LogiFlow-RL package and dependencies
2. **Verified** the 12-node environment loads correctly
3. **Ran `train_grpo.py`** which is the production training pipeline:
   - Built a dataset of prompts from environment rollouts
   - Validated the reward function (sanity check)
   - Evaluated `round_robin` and `heuristic` baselines
   - Trained Qwen2.5-0.5B with GRPO + LoRA for 200 steps
   - Evaluated the trained model on all 3 tasks
   - Saved 6 artifact files for the README and blog
4. **Displayed** the reward curve, improvement JSON, and before/after comparison

## Files generated

| File | Purpose |
|------|---------|
| `reward_curve.png` | Training reward over time — embed in README |
| `evaluation_before.csv` | Baseline scores before training |
| `evaluation_after.csv` | Trained model scores |
| `evaluation_summary.csv` | Both phases combined |
| `evaluation_summary.json` | Same data as JSON |
| `improvement.json` | Score deltas |

